# Shapes and Colors Prediction Challenge 🟥🔺🔵

**Author:** Patrick Chen  
**Duration:** 72 hours  
**Submission Type:** Take-home assignment  
**Evaluation:** Based on insight, analysis, and model development

## Objective

The goal of this challenge is to detect and identify geometric shapes and their colors in synthetic images. Each image may contain multiple instances of:

- Shapes: `circle`, `square`, `triangle`
- Colors: `red`, `green`, `blue`

The model should predict all `(shape, color)` pairs present in each image.

## Dataset Overview

- Training Dataset provided in CSV format with image paths and ground truth labels.
- Testing Dataset only includes image paths, no labels.
- Each label is a list of shape-color tuples.
- Sample image resolution: 128x128 RGB for model adaption.
- No overlapping objects; random positions, sizes, and rotations.

```python
class ShapesColorsDataset(Dataset):
    def __init__(self, csv_file, root_dir, mode='train', transform=None):
        self.annotations = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.mode = mode
        self.transform = transform or transforms.Compose([
            transforms.Resize((128, 128)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        img_rel_path = self.annotations.iloc[idx, 0]
        img_path = os.path.join(self.root_dir, img_rel_path)
        image = Image.open(img_path).convert("RGB")

        if self.mode == 'test':
            return self.transform(image), img_rel_path  # test mode

        labels = ast.literal_eval(self.annotations.iloc[idx, 1])  # list of (shape, color)
        target_vector = self.encode_labels(labels)
        return self.transform(image), torch.tensor(target_vector).float()        

    def encode_labels(self, labels):
        # 3 shapes × 3 colors = 9 possible (shape, color) combos
        classes = [
            ('circle', 'red'), ('circle', 'green'), ('circle', 'blue'),
            ('square', 'red'), ('square', 'green'), ('square', 'blue'),
            ('triangle', 'red'), ('triangle', 'green'), ('triangle', 'blue')
        ]
        target = [0] * len(classes)
        for label in labels:
            if label in classes:
                target[classes.index(label)] = 1
        return target
```

## Data Preprocessing
Each input image is first resized to a fixed resolution of 128×128 pixels to ensure uniformity across the dataset, and then converted into a normalized tensor for model compatibility. The custom `ShapesColorsDataset` class loads image-label pairs from the CSV file, where labels are parsed into binary multi-hot vectors representing the presence of each shape-color combination. This preprocessing pipeline enables efficient batch loading and consistent input formatting for training and evaluation.

You can use this:
```python
from dataset import ShapesColorsDataset
from torchvision import transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

dataset = ShapesColorsDataset("train_v2.csv", "dataset_v2", transform=transform)
```
---

Or you can just let the `ShapesColorsDataset` do it:

```python

class ShapesColorsDataset(Dataset):
    def __init__(self, csv_file, root_dir, mode='train', transform=None):
        self.annotations = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.mode = mode
        self.transform = transform or transforms.Compose([
            transforms.Resize((128, 128)),
            transforms.ToTensor(),
        ])

dataset = ShapesColorsDataset("train_v2.csv", "datraset_v2")
```

## Model Architecture

![1](./figure/Resnet18.png)

The model used for this task is a lightweight multi-label classifier based on a ResNet-18 backbone. The `ShapeColorClassifier` leverages the convolutional feature extraction capabilities of `torchvision.models.resnet18`, initialized with random weights (`weights=None`) to prevent bias from pretraining on unrelated datasets. The final fully connected layer is replaced with a new linear layer that outputs a 9-dimensional vector, corresponding to the 9 possible (shape, color) combinations (3 shapes × 3 colors). The output logits are later passed through a sigmoid function during loss calculation to support multi-label classification, where each class is predicted independently. This simple yet effective architecture balances model capacity and computational efficiency, making it well-suited for synthetic geometric object detection.

## Evaluation Metric: Jaccard Similarity

```python
def jaccard_index(pred_set, true_set):
    intersection = len(pred_set & true_set)
    union = len(pred_set | true_set)
    return intersection / union if union > 0 else 1.0
```

The primary metric used to evaluate model predictions is the **Jaccard similarity**, also known as the **Intersection over Union (IoU)** for sets.

Given:
- **A**: the predicted set of `(shape, color)` tuples
- **B**: the ground-truth set of `(shape, color)` tuples

The **Jaccard similarity** is defined as:

$$
J(A, B) = \frac{|A \cap B|}{|A \cup B|}
$$

### Interpretation:

- **J = 1.0** → Perfect prediction (all shapes and colors matched exactly)
- **J = 0.0** → Completely incorrect (no overlap between predicted and true labels)
- Values between 0 and 1 indicate partial correctness

This metric is well-suited for **multi-label classification** tasks like this one, where each image can contain multiple objects and classes are not mutually exclusive.

## Training Strategy

- Optimizer: AdamW
- Max LR: 8e-4
- Min LR: 5e-4
- Weight Decay 3e-4
- Scheduler: Custom with warmup + cosine annealing + optional final decay
- Loss: BCEWithLogitsLoss
- Training Epochs: 500 epochs
- Batch Size: 128
- Val split: 20%
- Metric: Jaccard Similarity
- Validation on validation dataset every 100 steps using Jaccard Similarity metric and validation loss during training.
- Keep tracking and saving the checkpoint with the highest Jaccard Similarity.

running command:
```bash
CUDA_VISIBLE_DEVICES=2 python ./train.py --train_csv /data/patrick/kaggle/all_shapes_and_colors/train_v2.csv --train_dir /data/patrick/kaggle/all_shapes_and_colors/dataset_v2 --num_epochs 500 --batch_size 128 --max_lr 8e-4 --min_lr 5e-5 --weight_decay 3e-4 --out_path ./output_bz128_warm1500_nepoch500_maxlr8e-4_minlr5e-5_wdcay_3e-4 --warmup_steps 1500
```

training screenshot:

![1](./figure/train_ep1.png)

.
.
.
![2](./figure/train_ep300.png)

.
.
.

![3](./figure/train_ep500.png)



## Results and Plots

### Training & Validation Loss

![loss](./output_bz128_warm1500_nepoch500_maxlr8e-4_minlr5e-5_wdcay_3e-4/training_validation_loss_vs_steps.png)

### Jaccard Similarity Metric over Validation

![jaccard](./output_bz128_warm1500_nepoch500_maxlr8e-4_minlr5e-5_wdcay_3e-4/metric_jaccard_vs_steps.png)

### Learning Rate Schedule

![lr](./output_bz128_warm1500_nepoch500_maxlr8e-4_minlr5e-5_wdcay_3e-4/learning_rate_vs_steps.png)

### Gradient Norms

![grad](./output_bz128_warm1500_nepoch500_maxlr8e-4_minlr5e-5_wdcay_3e-4/grad_norm_vs_steps.png)

## Surprising Found

- **Empty Cases:**  
  During evaluation, we discovered a number of test images with **no objects present**, and the labels provided in the dataset are all zeros, as shown figures below. These cases initially caused instability in the validation loss (due to the nature of `BCEWithLogitsLoss` expecting at least one positive/negative class). To mitigate this, we applied a filter to **skip samples where the label vector sums to zero**, effectively stabilizing loss computation and improving model robustness.

  For example:

  ![0](./output_test_no_skip_empty/high_loss_443.png)

  ![0](./figure/all_zero_case.png)

  Skip empty code:
  ```python
        for batch_idx, (images, labels) in enumerate(pbar):
            images, labels = images.cuda(), labels.cuda()
            non_empty_mask = labels.sum(dim=1) > 0
            if non_empty_mask.sum() == 0:
                continue  # skip this batch entirely
            # Filter images and labels
            images = images[non_empty_mask]
            labels = labels[non_empty_mask]
    ```


- **High-Loss Outliers:**  
  Some validation images triggered **unexpectedly high loss values** (e.g., >10), even after convergence. Upon inspection, these were often due to the model **confidently predicting incorrect shape-color combinations** for certain rare or ambiguous configurations. These samples were logged for visualization and analysis, providing valuable insight into the model's uncertainty.

- **Sharp Jaccard Drops:**  
  The validation Jaccard score exhibited sudden dips, often aligned with learning rate phase transitions, gradient esplosion or outlier batches. This highlighted the importance of **monitoring gradient norms and adapting scheduler decay behavior**, especially when using a cosine warmup strategy.

## Evaluation

- **Training Dataset**: Random split by 80% from the provided training dataset, which is 4000 images and labels.
- **Validation Dataset**: Random split by 20% from the provided training dataset, which is 1000 images and labels.
- **Final Jaccard Score** (on validation): ~0.99, with best checkpoint saved at 0.9992.
- **Training Loss**: converged to near 0 at epoch 500.
- **Validation Loss**: stabilized after LR warmup, and converged to nearly 0.003 at epoch 500.
- **Outliers**: handled by skipping empty-label samples and logging high-loss examples.
- **Loss Curve Analysis**: 

The training and validation loss curves show several spikes during early and mid-training stages (before step ~12,000). These spikes are caused by **hard cases** in the validation set — particularly images with ambiguous shapes or rare combinations that can lead to unstable predictions under `BCEWithLogitsLoss`.

Despite these outliers, the model steadily improves, and **after 12,000 steps**, both training and validation losses **stabilize at a low plateau**, indicating convergence. Importantly, the **gap between training and validation loss remains narrow**, suggesting that the model **generalizes well and is not overfitting**.

This stable behavior in later epochs aligns with high Jaccard scores and consistent predictions, validating the effectiveness of the learning rate schedule and regularization strategy.

## Insights & Takeaways

- The task is multi-label, not multi-class → BCE loss is appropriate.
- Mini-Batch Gradient Descent generalization helped model learn flatter minima, avoiding overfitting.
- Most performance gains came from:
  - Custom scheduler
  - Careful loss filtering (handling empty-label samples)
  - Monitoring gradient norm, training/validation loss, validation Jaccard Metric, and learning rate for stability

## Submission

- Predictions submitted to Kaggle: `./output_bz128_warm1500_nepoch500_maxlr8e-4_minlr5e-5_wdcay_3e-4/submission.csv`
- Format verified as:
```csv
image_path,label
test_dataset/img_0.png,"[('circle', 'red'), ('square', 'blue')]"
```
- Predictions Screenshot:

![0](./figure/prediction_result.png)


## Codes/Outpus Locations

- URL: https://github.com/Coslate/All_Shapes_and_Colors
- Code: `train.py`, `eval.py`, `dataset.py`, `model.py`, `scheduler.py`
- Trained model checkpoints: `./output_bz128_warm1500_nepoch500_maxlr8e-4_minlr5e-5_wdcay_3e-4/9888_shape_color_model.pth`
- Final predictions CSV: `./output_bz128_warm1500_nepoch500_maxlr8e-4_minlr5e-5_wdcay_3e-4/submission.csv`
- This report: `./Shapes_Colors_Report.ipynb`

## Thank You

This challenge provided a valuable opportunity to exercise full deep learning workflow — from data to deployment.

**Kind regards,**  
Patrick Chen